# Customer Churn Prediction System
### End-to-End Machine Learning Project

**Goal:** Predict whether a customer is likely to leave (churn) or stay, using their
demographic info, usage behavior, and payment history.

This notebook is being built **phase by phase**. This first part covers:
- **Task 1** — Load and Understand the Dataset
- **Task 2** — Basic Data Inspection

Later phases will add cleaning, EDA, encoding, model training, evaluation, tuning,
and the final prediction system.

## Task 1: Load and Understand the Dataset

First we load the required libraries and the dataset itself, then take a first look
at what's inside — the first few rows, the last few rows, a random sample, and the
overall size of the data.

In [1]:
import pandas as pd
import numpy as np

# Display settings so wide tables don't get cut off
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('data/customer_churn.csv')
print("Dataset loaded successfully.")

Dataset loaded successfully.


In [2]:
# First 5 records
df.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0


In [3]:
# Last 5 records
df.tail()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
64369,64370,45,Female,33,12,6,21,Basic,Quarterly,947,14,1
64370,64371,37,Male,6,1,5,22,Standard,Annual,923,9,1
64371,64372,25,Male,39,14,8,30,Premium,Monthly,327,20,1
64372,64373,50,Female,18,19,7,22,Standard,Monthly,540,13,1
64373,64374,52,Female,45,15,9,25,Standard,Monthly,696,22,1


In [4]:
# A random sample of 5 records (useful to spot-check data that isn't just at the start/end)
df.sample(5, random_state=42)

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
15476,15477,55,Male,20,24,4,6,Standard,Monthly,635,25,0
34666,34667,28,Male,27,30,4,5,Premium,Quarterly,631,10,0
50474,50475,65,Female,60,17,7,16,Premium,Quarterly,314,1,1
7984,7985,53,Male,47,16,8,7,Premium,Annual,527,13,0
20227,20228,32,Male,56,5,7,15,Premium,Annual,236,25,0


In [5]:
# Total number of customers (rows) and columns
n_rows, n_cols = df.shape
print(f"Total customers (rows): {n_rows}")
print(f"Total columns: {n_cols}")
print(f"\nColumn names: {list(df.columns)}")

Total customers (rows): 64374
Total columns: 12

Column names: ['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction', 'Churn']


### What each column represents

Grouping the columns by what kind of information they capture makes the dataset much
easier to reason about later during EDA and modeling:

| Category | Columns |
|---|---|
| **Identifier** (not predictive — will be dropped) | `CustomerID` |
| **Demographics** | `Age`, `Gender` |
| **Usage behavior** | `Tenure`, `Usage Frequency`, `Last Interaction` |
| **Support / payment behavior** | `Support Calls`, `Payment Delay` |
| **Subscription info** | `Subscription Type`, `Contract Length`, `Total Spend` |
| **Target variable** | `Churn` (1 = customer churned, 0 = customer stayed) |

**`Churn` is the target variable** — it's what we're trying to predict. Every other
column is a potential *feature* (input) the model can learn from, except `CustomerID`,
which is just a label with no real behavioral meaning.

In [6]:
feature_groups = {
    "Identifier (drop before modeling)": ["CustomerID"],
    "Demographics": ["Age", "Gender"],
    "Usage behavior": ["Tenure", "Usage Frequency", "Last Interaction"],
    "Support / payment behavior": ["Support Calls", "Payment Delay"],
    "Subscription info": ["Subscription Type", "Contract Length", "Total Spend"],
    "Target": ["Churn"],
}

for group, cols in feature_groups.items():
    print(f"{group}: {cols}")

Identifier (drop before modeling): ['CustomerID']
Demographics: ['Age', 'Gender']
Usage behavior: ['Tenure', 'Usage Frequency', 'Last Interaction']
Support / payment behavior: ['Support Calls', 'Payment Delay']
Subscription info: ['Subscription Type', 'Contract Length', 'Total Spend']
Target: ['Churn']


## Task 2: Basic Data Inspection

Before touching the data, we need to fully understand its shape and quality: data
types, unique values, missing values, duplicates, and summary statistics. This tells
us what cleaning (Task 3) will actually be needed.

In [7]:
# Rows, columns, column names, and data types all in one summary
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64374 entries, 0 to 64373
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   CustomerID         64374 non-null  int64
 1   Age                64374 non-null  int64
 2   Gender             64374 non-null  str  
 3   Tenure             64374 non-null  int64
 4   Usage Frequency    64374 non-null  int64
 5   Support Calls      64374 non-null  int64
 6   Payment Delay      64374 non-null  int64
 7   Subscription Type  64374 non-null  str  
 8   Contract Length    64374 non-null  str  
 9   Total Spend        64374 non-null  int64
 10  Last Interaction   64374 non-null  int64
 11  Churn              64374 non-null  int64
dtypes: int64(9), str(3)
memory usage: 5.9 MB


In [8]:
# Number of unique values per column
df.nunique()

CustomerID           64374
Age                     48
Gender                   2
Tenure                  60
Usage Frequency         30
Support Calls           11
Payment Delay           31
Subscription Type        3
Contract Length          3
Total Spend            901
Last Interaction        30
Churn                    2
dtype: int64

In [9]:
# Summary statistics for numerical columns
df.describe()

,CustomerID,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn
count,64374.000000,64374.000000,64374.000000,64374.000000,64374.000000,64374.000000,64374.000000,64374.000000,64374.000000
mean,32187.500000,41.970982,31.994827,15.080234,5.400690,17.133952,541.023379,15.498850,0.473685
std,18583.317451,13.924911,17.098234,8.816470,3.114005,8.852211,260.874809,8.638436,0.499311
min,1.000000,18.000000,1.000000,1.000000,0.000000,0.000000,100.000000,1.000000,0.000000
25%,16094.250000,30.000000,18.000000,7.000000,3.000000,10.000000,313.000000,8.000000,0.000000
50%,32187.500000,42.000000,33.000000,15.000000,6.000000,19.000000,534.000000,15.000000,0.000000
75%,48280.750000,54.000000,47.000000,23.000000,8.000000,25.000000,768.000000,23.000000,1.000000
max,64374.000000,65.000000,60.000000,30.000000,10.000000,30.000000,1000.000000,30.000000,1.000000


In [10]:
# Missing values per column
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing values in dataset: {missing.sum()}")

Missing values per column:
CustomerID           0
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64

Total missing values in dataset: 0


In [11]:
# Duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate records: {duplicate_count}")

Number of duplicate records: 0


In [12]:
# Distribution of the target variable: Churn
churn_counts = df['Churn'].value_counts()
churn_percent = df['Churn'].value_counts(normalize=True) * 100

print("Churn counts:")
print(churn_counts)
print("\nChurn percentage:")
print(churn_percent.round(2))

Churn counts:
Churn
0    33881
1    30493
Name: count, dtype: int64

Churn percentage:
Churn
0    52.63
1    47.37
Name: proportion, dtype: float64


### Is the target balanced or imbalanced?

Run the cell above and compare the two counts:
- If one class (say, "stayed") massively outnumbers the other (say, 90% vs 10%), the
  target is **imbalanced**. A lazy model could just predict "no churn" every single
  time and still score ~90% accuracy — while being completely useless for actually
  catching churners.
- If the split is closer to 50/50 (or within roughly 60/40), it's **reasonably
  balanced**, and plain accuracy is a fairer metric — though we'll still track
  precision/recall/F1 later since, in churn prediction, *missing* a customer who
  churns is usually more costly than a false alarm.

**Why this matters before training:** class balance decides which evaluation metrics
you can trust (Task 12) and whether you'll need techniques like class weighting or
resampling. This is exactly why Task 2 always comes before jumping into modeling.

---
### ✅ Phase 1 checkpoint

At this point you should be able to answer, in your own words:
1. How many customers and columns does the dataset have?
2. What does each column represent, and which group (demographics / usage /
   payment / subscription / target) does it belong to?
3. Is there any missing data or duplicate records?
4. Is the `Churn` target balanced or imbalanced, and why does that matter?

Once you're comfortable with these, we move to **Task 3 (Data Cleaning)** and
**Task 4 (Exploratory Data Analysis)** next.